In [1]:
import pandas as pd
import dask.dataframe as dd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import LabelEncoder, StandardScaler
import category_encoders as ce
import gc

DATA_FOLDER ='/mnt/d/TON_IoT_Dataset'


In [12]:
df = dd.read_csv(Path(DATA_FOLDER) / 'combined.csv', dtype={'src_bytes': 'object', 'uid': 'object'})

df.head()

,ts,src_ip,src_port,dst_ip,dst_port,proto,service,duration,src_bytes,dst_bytes,...,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,weird_notice,label,type,uid,time_group
0,1554198358,3.122.49.24,1883,192.168.1.152,52976,tcp,-,80549.530260,1762852,41933215,...,-,-,-,bad_TCP_checksum,-,F,0,normal,<NA>,0
1,1554198358,192.168.1.79,47260,192.168.1.255,15600,udp,-,0.000000,0,0,...,-,-,-,-,-,-,0,normal,<NA>,0
2,1554198359,192.168.1.152,1880,192.168.1.152,51782,tcp,-,0.000000,0,0,...,-,-,-,bad_TCP_checksum,-,F,0,normal,<NA>,0
3,1554198359,192.168.1.152,34296,192.168.1.152,10502,tcp,-,0.000000,0,0,...,-,-,-,-,-,-,0,normal,<NA>,0
4,1554198362,192.168.1.152,46608,192.168.1.190,53,udp,dns,0.000549,0,298,...,-,-,-,bad_UDP_checksum,-,F,0,normal,<NA>,0


In [13]:
df = df.sort_values('ts')

# time split
split_secs = 20
start_time = df['ts'].min().compute()
df['time_group'] = (df['ts'] - start_time) // split_secs

# construct src and dst, drop unnecessary columns
df['src'] = df['src_ip'].astype('str') + ':' + df['src_port'].astype('str')
df['dst'] = df['dst_ip'].astype('str') + ':' + df['dst_port'].astype('str')
df = df.drop(columns=['uid', 'ts', 'src_ip', 'dst_ip', 'src_port','dst_port','http_uri','weird_name','weird_addl','weird_notice','dns_query','ssl_subject','ssl_issuer','http_user_agent','label'])

# order columns
cols = ['src', 'dst'] + [col for col in df.columns if col not in ['src', 'dst']]
df = df[cols]

# rename "type" to "label"
df = df.rename(columns={"type": "label"})

# fix mislabeled values in 'src_bytes' column
df['src_bytes'] = df['src_bytes'].map_partitions(
    lambda s: s.astype('object').replace('0.0.0.0', '0').astype(int),
    meta=('src_bytes', 'int64')
)

df.head(1000)

,src,dst,proto,service,duration,src_bytes,dst_bytes,conn_state,missed_bytes,src_pkts,...,http_method,http_referrer,http_version,http_request_body_len,http_response_body_len,http_status_code,http_orig_mime_types,http_resp_mime_types,label,time_group
1,192.168.1.79:47260,192.168.1.255:15600,udp,-,0.000000,0,0,S0,0,1,...,-,-,-,0,0,0,-,-,normal,0
0,3.122.49.24:1883,192.168.1.152:52976,tcp,-,80549.530260,1762852,41933215,OTH,0,252181,...,-,-,-,0,0,0,-,-,normal,0
2,192.168.1.152:1880,192.168.1.152:51782,tcp,-,0.000000,0,0,OTH,0,0,...,-,-,-,0,0,0,-,-,normal,0
3,192.168.1.152:34296,192.168.1.152:10502,tcp,-,0.000000,0,0,OTH,0,0,...,-,-,-,0,0,0,-,-,normal,0
4,192.168.1.152:46608,192.168.1.190:53,udp,dns,0.000549,0,298,SHR,0,0,...,-,-,-,0,0,0,-,-,normal,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,192.168.1.79:50713,192.168.1.255:15600,udp,-,0.000000,0,0,S0,0,1,...,-,-,-,0,0,0,-,-,normal,75
996,192.168.1.152:34296,192.168.1.152:10502,tcp,-,0.000000,0,0,OTH,0,0,...,-,-,-,0,0,0,-,-,normal,75
997,192.168.1.152:1880,192.168.1.152:51782,tcp,-,0.000000,0,0,OTH,0,0,...,-,-,-,0,0,0,-,-,normal,75
998,192.168.1.152:34296,192.168.1.152:10502,tcp,-,0.000000,0,0,OTH,0,0,...,-,-,-,0,0,0,-,-,normal,75


In [14]:
# check for inf and nan values
numeric_df = df.select_dtypes(include=[np.number])
rows_with_invalid = numeric_df.map_partitions(lambda part: (np.isinf(part) | part.isna()).any(axis=1), meta=bool)
print(f'Number of invalid values: {int(rows_with_invalid.sum().compute())}')

Number of invalid values: 0


In [ ]:
# label time groups
def label_group_pandas(group):
    unique_labels = set(group['label'].unique())
    if len(unique_labels) == 1:
        label = unique_labels.pop()
    elif 'normal' in unique_labels and len(unique_labels) == 2:
        unique_labels.remove('normal')
        label = unique_labels.pop()
    else:
        label = 'unknown'
    return pd.DataFrame({'time_group': [group.name], 'label': [label]})

label_df = df.groupby('time_group').apply(
    label_group_pandas,
    meta={'time_group': 'int64', 'label': 'object'}
).compute()

label_df = label_df.reset_index(drop=True).sort_values('time_group')

# assign each time group into train/test
def split_train_test_pandas(group):
    n = len(group)
    train_end = int(n * 0.8)
    group['type'] = ['train'] * train_end + ['test'] * (n - train_end)
    return group

labeled_with_split = label_df.groupby('label', group_keys=False).apply(split_train_test_pandas)
labeled_with_split = labeled_with_split[labeled_with_split['label'] != 'unknown']

# save output
output_path = Path(DATA_FOLDER) / "labels.csv"
labeled_with_split.to_csv(output_path, index=False)
print(f"Saved to {output_path}")


Saved to /mnt/d/TON_IoT_Dataset/labels.csv


/tmp/ipykernel_12420/2484909722.py:27: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  labeled_with_split = label_df.groupby('label', group_keys=False).apply(split_train_test_pandas)


In [16]:
unique_labels = df['label'].unique().compute()
le = LabelEncoder()
le.fit(unique_labels)
label_map = dict(zip(le.classes_, le.transform(le.classes_)))
df['label'] = df['label'].map(label_map, meta=('label', 'int64'))

df.head(1000)

,src,dst,proto,service,duration,src_bytes,dst_bytes,conn_state,missed_bytes,src_pkts,...,http_method,http_referrer,http_version,http_request_body_len,http_response_body_len,http_status_code,http_orig_mime_types,http_resp_mime_types,label,time_group
1,192.168.1.79:47260,192.168.1.255:15600,udp,-,0.000000,0,0,S0,0,1,...,-,-,-,0,0,0,-,-,5,0
0,3.122.49.24:1883,192.168.1.152:52976,tcp,-,80549.530260,1762852,41933215,OTH,0,252181,...,-,-,-,0,0,0,-,-,5,0
2,192.168.1.152:1880,192.168.1.152:51782,tcp,-,0.000000,0,0,OTH,0,0,...,-,-,-,0,0,0,-,-,5,0
3,192.168.1.152:34296,192.168.1.152:10502,tcp,-,0.000000,0,0,OTH,0,0,...,-,-,-,0,0,0,-,-,5,0
4,192.168.1.152:46608,192.168.1.190:53,udp,dns,0.000549,0,298,SHR,0,0,...,-,-,-,0,0,0,-,-,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,192.168.1.79:50713,192.168.1.255:15600,udp,-,0.000000,0,0,S0,0,1,...,-,-,-,0,0,0,-,-,5,75
996,192.168.1.152:34296,192.168.1.152:10502,tcp,-,0.000000,0,0,OTH,0,0,...,-,-,-,0,0,0,-,-,5,75
997,192.168.1.152:1880,192.168.1.152:51782,tcp,-,0.000000,0,0,OTH,0,0,...,-,-,-,0,0,0,-,-,5,75
998,192.168.1.152:34296,192.168.1.152:10502,tcp,-,0.000000,0,0,OTH,0,0,...,-,-,-,0,0,0,-,-,5,75


In [17]:
le.classes_

array(['backdoor', 'ddos', 'dos', 'injection', 'mitm', 'normal',
       'password', 'ransomware', 'scanning', 'xss'], dtype=object)

In [18]:
idx_label_df = pd.DataFrame(le.classes_)
idx_label_df.columns = ['label']
idx_label_df.to_csv(Path(DATA_FOLDER) / 'idx_label.csv', index=True)

In [19]:
df['label'].value_counts().compute()

label
9    2108944
2    3375328
3     452659
5     796380
7      72805
6    1718568
4       1052
0     508116
1    6165008
8    7140161
Name: count, dtype: int64

In [20]:
df.to_csv(Path(DATA_FOLDER) / 'data-checkpoint1.csv', index=False, single_file=True)

['/mnt/d/TON_IoT_Dataset/data-checkpoint1.csv']

In [ ]:
# in case of limited memory, reset kernel here

df = pd.read_csv(Path(DATA_FOLDER) / 'data-checkpoint1.csv')
label_df = pd.read_csv(Path(DATA_FOLDER) / 'labels.csv')

train_idx = set(label_df[label_df['type'] == 'train']['time_group'])
test_idx = set(label_df[label_df['type'] == 'test']['time_group'])

train_df = df[df['time_group'].isin(train_idx)]
test_df = df[df['time_group'].isin(test_idx)]

print(train_df.shape[0] + test_df.shape[0])

test_df.to_csv(Path(DATA_FOLDER) / 'test-data-checkpoint1.csv', index=False)

del df
del test_df
gc.collect()

22339021


20

In [3]:
categorical_cols = [
    'proto','service','conn_state','dns_qclass','dns_qtype','dns_rcode','dns_AA','dns_RD',
    'dns_RA','dns_rejected','ssl_version','ssl_cipher','ssl_resumed','http_referrer','ssl_established',
    'http_method','http_version','http_status_code','http_orig_mime_types','http_resp_mime_types',
    'http_trans_depth'
]
target_col = 'label'

# save memory by processing data col by col
encoders = {}
for col in categorical_cols:
    encoder = ce.TargetEncoder(cols=[col])
    encoder.fit(train_df[[col]], train_df[target_col])
    train_df[col] = encoder.transform(train_df[[col]])[col]
    encoders[col] = encoder

train_df.head(5000)

,src,dst,proto,service,duration,src_bytes,dst_bytes,conn_state,missed_bytes,src_pkts,...,http_method,http_referrer,http_version,http_request_body_len,http_response_body_len,http_status_code,http_orig_mime_types,http_resp_mime_types,label,time_group
0,3.122.49.24:1883,192.168.1.152:52976,4.660132,4.400649,80549.530260,1762852,41933215,6.272580,0,252181,...,4.648917,4.645753,4.648868,0,0,4.648868,4.645655,4.645455,5,0
1,192.168.1.79:47260,192.168.1.255:15600,4.464285,4.400649,0.000000,0,0,7.643964,0,1,...,4.648917,4.645753,4.648868,0,0,4.648868,4.645655,4.645455,5,0
2,192.168.1.152:1880,192.168.1.152:51782,4.660132,4.400649,0.000000,0,0,6.272580,0,0,...,4.648917,4.645753,4.648868,0,0,4.648868,4.645655,4.645455,5,0
3,192.168.1.152:34296,192.168.1.152:10502,4.660132,4.400649,0.000000,0,0,6.272580,0,0,...,4.648917,4.645753,4.648868,0,0,4.648868,4.645655,4.645455,5,0
4,192.168.1.152:46608,192.168.1.190:53,4.464285,4.427519,0.000549,0,298,1.423904,0,0,...,4.648917,4.645753,4.648868,0,0,4.648868,4.645655,4.645455,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,192.168.1.152:34296,192.168.1.152:10502,4.660132,4.400649,0.000000,0,0,6.272580,0,0,...,4.648917,4.645753,4.648868,0,0,4.648868,4.645655,4.645455,5,366
4996,192.168.1.152:1880,192.168.1.152:51782,4.660132,4.400649,0.000000,0,0,6.272580,0,0,...,4.648917,4.645753,4.648868,0,0,4.648868,4.645655,4.645455,5,366
4997,192.168.1.79:35156,192.168.1.255:15600,4.464285,4.400649,0.000000,0,0,7.643964,0,1,...,4.648917,4.645753,4.648868,0,0,4.648868,4.645655,4.645455,5,366
4998,192.168.1.152:34296,192.168.1.152:10502,4.660132,4.400649,0.000000,0,0,6.272580,0,0,...,4.648917,4.645753,4.648868,0,0,4.648868,4.645655,4.645455,5,366


In [4]:
scalers = {}

cols_to_norm = list(set(train_df.columns) - {'src', 'dst', 'label', 'time_group'})

for col in cols_to_norm:
    scaler = StandardScaler()
    scaled_col = scaler.fit_transform(train_df[[col]])
    train_df[col] = scaled_col
    scalers[col] = scaler

train_df.to_csv(Path(DATA_FOLDER) / 'train-data-checkpoint1.csv', index=False)
train_df.head(5000)

,src,dst,proto,service,duration,src_bytes,dst_bytes,conn_state,missed_bytes,src_pkts,...,http_method,http_referrer,http_version,http_request_body_len,http_response_body_len,http_status_code,http_orig_mime_types,http_resp_mime_types,label,time_group
0,3.122.49.24:1883,192.168.1.152:52976,0.212059,-0.301861,915.831824,0.020139,1.168430,0.675081,-0.007471,1228.262664,...,0.034097,-0.000342,0.0354,-0.003549,-0.000638,0.029281,-0.008506,-0.016452,5,0
1,192.168.1.79:47260,192.168.1.255:15600,-2.676460,-0.301861,-0.096970,-0.027123,-0.023370,1.244161,-0.007471,-0.015245,...,0.034097,-0.000342,0.0354,-0.003549,-0.000638,0.029281,-0.008506,-0.016452,5,0
2,192.168.1.152:1880,192.168.1.152:51782,0.212059,-0.301861,-0.096970,-0.027123,-0.023370,0.675081,-0.007471,-0.020115,...,0.034097,-0.000342,0.0354,-0.003549,-0.000638,0.029281,-0.008506,-0.016452,5,0
3,192.168.1.152:34296,192.168.1.152:10502,0.212059,-0.301861,-0.096970,-0.027123,-0.023370,0.675081,-0.007471,-0.020115,...,0.034097,-0.000342,0.0354,-0.003549,-0.000638,0.029281,-0.008506,-0.016452,5,0
4,192.168.1.152:46608,192.168.1.190:53,-2.676460,-0.268767,-0.096963,-0.027123,-0.023361,-1.336964,-0.007471,-0.020115,...,0.034097,-0.000342,0.0354,-0.003549,-0.000638,0.029281,-0.008506,-0.016452,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,192.168.1.152:34296,192.168.1.152:10502,0.212059,-0.301861,-0.096970,-0.027123,-0.023370,0.675081,-0.007471,-0.020115,...,0.034097,-0.000342,0.0354,-0.003549,-0.000638,0.029281,-0.008506,-0.016452,5,366
4996,192.168.1.152:1880,192.168.1.152:51782,0.212059,-0.301861,-0.096970,-0.027123,-0.023370,0.675081,-0.007471,-0.020115,...,0.034097,-0.000342,0.0354,-0.003549,-0.000638,0.029281,-0.008506,-0.016452,5,366
4997,192.168.1.79:35156,192.168.1.255:15600,-2.676460,-0.301861,-0.096970,-0.027123,-0.023370,1.244161,-0.007471,-0.015245,...,0.034097,-0.000342,0.0354,-0.003549,-0.000638,0.029281,-0.008506,-0.016452,5,366
4998,192.168.1.152:34296,192.168.1.152:10502,0.212059,-0.301861,-0.096970,-0.027123,-0.023370,0.675081,-0.007471,-0.020115,...,0.034097,-0.000342,0.0354,-0.003549,-0.000638,0.029281,-0.008506,-0.016452,5,366


In [5]:
del train_df
gc.collect()

input_path = Path(DATA_FOLDER) / 'train-data-checkpoint1.csv'
output_path = Path(DATA_FOLDER) / 'X_train.csv'

open(output_path, 'w').close()

# Chunked processing and writing
first_chunk = True

for chunk in pd.read_csv(input_path, chunksize=100_000):
    # Compute features column
    chunk['features'] = chunk[cols_to_norm].values.tolist()

    # Select required columns
    output_chunk = chunk[['src', 'dst', 'features', 'time_group']]

    # Write to disk
    output_chunk.to_csv(output_path, mode='a', index=False, header=first_chunk)
    first_chunk = False  # only write header once

    del chunk, output_chunk
    gc.collect()

In [6]:
test_df = pd.read_csv(Path(DATA_FOLDER) / 'test-data-checkpoint1.csv')

for col in categorical_cols:
    encoder = encoders[col]
    test_df[col] = encoder.transform(test_df[[col]])[col]

for col in cols_to_norm:
    test_df[col] = scalers[col].transform(test_df[[col]])

test_df['features'] = test_df[cols_to_norm].values.tolist()

test_df.to_csv(Path(DATA_FOLDER) / 'X_test.csv', columns=['src', 'dst', 'features', 'time_group'], index=False)